# ECE 539 experiment 005: B0–B3 input comparison
Run cells in order. First select **Runtime → Change runtime type → GPU**. Large artifacts persist in Google Drive; canonical Test is not accessed. See `COLAB.md` for failure/restart policy.

In [ ]:
!nvidia-smi

In [ ]:
%cd /content
!if [ ! -d food-nutrition/.git ]; then git clone --branch 'merge-A&B' --single-branch https://github.com/xychen5165/ECE539-Grp5-FoodNutritionEst.git food-nutrition; fi
%cd /content/food-nutrition
!git checkout 'merge-A&B'
!git pull --ff-only origin 'merge-A&B'
!git rev-parse HEAD
!git diff --exit-code 112c26e98f99ee16984532b7798ba42c6bac6911 -- A_food_segmentation/sam_runtime.py B_nutrition_estimation/prepare_input_comparison.py B_nutrition_estimation/train_input_comparison.py B_nutrition_estimation/compare_input_conditions.py tests/test_b_input_comparison.py

In [ ]:
%pip install -q 'numpy>=1.26,<3' 'Pillow>=10,<13' open_clip_torch==2.24.0 opencv-python-headless==4.11.0.86 'segment-anything @ git+https://github.com/facebookresearch/segment-anything.git@dca509fe793f601edb92606367a655c15ac00fdf'

If Colab requests a runtime restart, restart now, rerun the GPU cell, and `%cd /content/food-nutrition` before continuing.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
drive.mount('/content/drive')
RUN_ROOT = Path('/content/drive/MyDrive/ECE539-FoodNutritionEst/005-b-input-comparison')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['RUN_ROOT'] = str(RUN_ROOT)
print(RUN_ROOT)

In [ ]:
!python B_nutrition_estimation/capture_colab_environment.py --output "$RUN_ROOT/environment.json" --run-root "$RUN_ROOT"
!cat "$RUN_ROOT/environment.json"

In [ ]:
!python B_nutrition_estimation/prepare_input_comparison.py download --root "$RUN_ROOT" --workers 12

In [ ]:
!mkdir -p "$RUN_ROOT/checkpoints"
![ -f "$RUN_ROOT/checkpoints/sam_vit_b_01ec64.pth" ] || wget -q --show-progress https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -O "$RUN_ROOT/checkpoints/sam_vit_b_01ec64.pth"
!sha256sum "$RUN_ROOT/checkpoints/sam_vit_b_01ec64.pth"

In [ ]:
!python B_nutrition_estimation/prepare_input_comparison.py segment --root "$RUN_ROOT" --sam-checkpoint "$RUN_ROOT/checkpoints/sam_vit_b_01ec64.pth" --device cuda

In [ ]:
!python B_nutrition_estimation/prepare_input_comparison.py features --root "$RUN_ROOT" --clip-cache "$RUN_ROOT/checkpoints" --batch-size 64 --device cuda

In [ ]:
!for CONDITION in B0 B1 B2 B3; do python B_nutrition_estimation/train_input_comparison.py "$CONDITION" --input-root "$RUN_ROOT" --output-root "$RUN_ROOT/results" || exit 1; done

In [ ]:
!python B_nutrition_estimation/compare_input_conditions.py --results "$RUN_ROOT/results" --output "$RUN_ROOT/comparison.json"
!python B_nutrition_estimation/package_colab_results.py --run-root "$RUN_ROOT" --output "$RUN_ROOT/005-small-results.zip"
!ls -lh "$RUN_ROOT/005-small-results.zip"

Download `005-small-results.zip` from Google Drive and attach it to the experiment task. It excludes raw images, masks, CLIP arrays and checkpoints.